# 03a — Synthetic Generation: Corrupted MusicXML → Rendered MIDI
**Score-Informed Dynamic Marking Adherence Evaluation — FYP Pipeline (fyp_try2)**

**The chosen path:** for every eligible piece, create *multiple* corrupted MusicXML files
(subtle one-level shifts as the majority, full inversions as a minority), then **render
each one to a MIDI performance** with human-like per-note jitter. Each rendered MIDI,
paired with the piece's *original* score, is a labelled *incorrect-adherence* example.

### Methodology problems addressed in this notebook
| # | Problem | How it is addressed here |
|---|---------|--------------------------|
| 2 | Class imbalance | Generates the missing negative (incorrect) examples |
| 3 | Disproportionate synthetic ratio | Per-piece count = `max(1, round(min(real, 10) × 0.3))` — proportional to the **capped** real count, so the ~30% share stays true after the cap at assembly |
| 5 | Trivially separable uniformity | ±`JITTER` per-note velocity jitter at render (set `JITTER=4`), and a **different seed per synthetic**. Set `JITTER=0` to build the no-jitter **ablation** set instead |
| 11 | Corruption too extreme | Majority severity = **one-level shifts** (p→pp, f→ff); full inversion kept as a minority; single-synthetic pieces always get a shift |
| 10 | Scores with no dynamics | Skipped explicitly (Bach urtext) |

### Division of labour
- **XML step** owns *what the wrong marking is* (severity decision, seeded per example)
- **Render step** owns *how loud + human wobble* — it renders the corrupted XML **faithfully**
  (no inversion logic in the MIDI code), writing only velocities onto a copy of
  `midi_score.mid`; onsets, durations, and pitches are untouched
- **Manifest JSON per example** records every corrupted marking (from → to), which becomes
  the per-marking label source at dataset assembly

### Wedge severity note
For wedges, a one-level "shift" has no symbolic equivalent, so subtle wedge errors are
made by flipping only a random **subset** of wedges (default 50%) in shift-mode examples,
leaving the rest correct — a partially-incorrect performance. Rendering flipped wedges
with a deliberately flattened slope is a further refinement option documented for the
experiments appendix.

### Two synthetic sets from one notebook
This notebook produces **one** synthetic set per run, chosen by the `JITTER` value:
run it with `JITTER = 4` to build the realistic `synthetic_jitter/` set, then run it
again with `JITTER = 0` to build the `synthetic_nojitter/` ablation set. Notebook 04
reads whichever set you point it at. The no-jitter set is expected to look *easier* to a
model — that is the point: comparing the two shows whether the jitter mitigation (Problem 5)
actually matters.

**Prerequisite:** run Notebook 01 first (needs `fyp_try2/cleaned_metadata.csv`).

In [ ]:
# @title Setup: install libraries, mount Drive, clone (n)ASAP
!pip install music21 pretty_midi -q

import os, sys, json, random, copy, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/fyp_try2')     # <-- everything saves here
    ASAP_ROOT = Path('/content/asap-dataset')          # session-local clone
else:
    BASE = Path('./fyp_try2')                          # local run (e.g. NVIDIA machine)
    ASAP_ROOT = Path('./asap-dataset')

if not ASAP_ROOT.exists():
    import subprocess
    print('Cloning (n)ASAP dataset (shallow, ~1-2 min)...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/CPJKU/asap-dataset.git', str(ASAP_ROOT)],
                   check=True)

BASE.mkdir(parents=True, exist_ok=True)
print('Drive base :', BASE)
print('ASAP root  :', ASAP_ROOT, '| exists:', ASAP_ROOT.exists())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive base : /content/drive/MyDrive/fyp_try2
ASAP root  : /content/asap-dataset | exists: True


## 2. Core functions
XML corruption (ElementTree, no music21 re-export), faithful render, and validation for both stages. All functions verified against real (n)ASAP pieces.

In [ ]:
# @title Core functions
import xml.etree.ElementTree as ET
import copy
import json
import random
from pathlib import Path

import numpy as np
import pretty_midi
from music21 import converter, dynamics as m21dynamics, spanner

# ----------------------------------------------------------------------
# Marking maps
# ----------------------------------------------------------------------
LEVEL_ORDER = ["ppp", "pp", "p", "mp", "mf", "f", "ff", "fff"]
CONSTANT_INVERSION = {
    "ppp": "fff", "pp": "ff", "p": "f", "mp": "mf",
    "mf": "mp", "f": "p", "ff": "pp", "fff": "ppp",
}
WEDGE_INVERSION = {"crescendo": "diminuendo", "diminuendo": "crescendo"}
VELOCITY_MAP = {
    "ppp": 16, "pp": 33, "p": 49, "mp": 64,
    "mf": 80, "f": 96, "ff": 112, "fff": 126,
}
DEFAULT_DYNAMIC = "mf"


def one_level_shift(marking,    ):
    """Shift a constant marking one adjacent level (random valid direction)."""
    if marking not in LEVEL_ORDER:
        return None
    i = LEVEL_ORDER.index(marking)
    options = [j for j in (i - 1, i + 1) if 0 <= j < len(LEVEL_ORDER)]
    return LEVEL_ORDER[rng.choice(options)]


# ----------------------------------------------------------------------
# Inspection (unchanged from earlier, verified against real ASAP)
# ----------------------------------------------------------------------
def scan_dynamics(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    constant, wedges = {}, {"crescendo": 0, "diminuendo": 0, "stop": 0}
    for dyn in root.iter("dynamics"):
        for child in dyn:
            constant[child.tag] = constant.get(child.tag, 0) + 1
    for wedge in root.iter("wedge"):
        wtype = wedge.get("type", "")
        if wtype in wedges:
            wedges[wtype] += 1
    return constant, wedges


def check_wedge_balance(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    issues = []
    for part in root.iter("part"):
        pid = part.get("id", "?")
        open_stack = {}
        for wedge in part.iter("wedge"):
            num = wedge.get("number", "1")
            wtype = wedge.get("type", "")
            if wtype in ("crescendo", "diminuendo"):
                if open_stack.get(num, 0) > 0:
                    issues.append(f"Part {pid}: overlapping wedge (number={num})")
                open_stack[num] = open_stack.get(num, 0) + 1
            elif wtype == "stop":
                if open_stack.get(num, 0) == 0:
                    issues.append(f"Part {pid}: stop with no open wedge (number={num})")
                else:
                    open_stack[num] -= 1
        for num, n_open in open_stack.items():
            if n_open > 0:
                issues.append(f"Part {pid}: {n_open} wedge(s) (number={num}) never stopped")
    return len(issues) == 0, issues


# ----------------------------------------------------------------------
# Generation: severity-mixed corruption of the raw XML
# ----------------------------------------------------------------------
def generate_synthetic_xml(src_path, dst_path, seed=42, mode="shift",
                           wedge_flip_prob=0.5):
    """
    Create a corrupted MusicXML file by editing the raw XML in place.
      mode='shift'  : constants -> one-level shift; wedges flipped with
                      probability wedge_flip_prob (subtle, near-boundary errors)
      mode='invert' : full mirror inversion of everything (extreme errors)
    Wedges are never added or moved, so no overlaps can be introduced and
    phrase-level spans stay identical to the original score.
    Returns a changelog (manifest): list of dicts, one per corrupted marking.
    """
    if mode not in ("shift", "invert"):
        raise ValueError(f"unknown mode: {mode}")
    rng = random.Random(seed)
    tree = ET.parse(src_path)
    root = tree.getroot()
    changelog = []

    # Constant markings
    for dyn in root.iter("dynamics"):
        for child in list(dyn):
            old = child.tag
            if mode == "invert":
                new = CONSTANT_INVERSION.get(old)
            else:
                new = one_level_shift(old, rng)
            if new and new != old:
                child.tag = new
                changelog.append({"kind": "constant", "from": old, "to": new})

    # Wedges (type flipped only; start/stop positions untouched)
    for wedge in root.iter("wedge"):
        old = wedge.get("type", "")
        new = WEDGE_INVERSION.get(old)
        if not new:
            continue
        do_flip = True if mode == "invert" else (rng.random() < wedge_flip_prob)
        if do_flip:
            wedge.set("type", new)
            changelog.append({"kind": "wedge", "from": old, "to": new})

    Path(dst_path).parent.mkdir(parents=True, exist_ok=True)
    tree.write(dst_path, encoding="UTF-8", xml_declaration=True)
    return changelog


# ----------------------------------------------------------------------
# XML validation (changelog-aware: works for both severity modes)
# ----------------------------------------------------------------------
def validate_pair(original_path, synthetic_path, changelog, check_notes=True):
    results = {}

    # Check 1: note content untouched (music21 parse - the slow check; in batch
    # runs it is enough to do this on the first synthetic of each piece, since
    # generation never touches notes and all synthetics share the same source)
    if check_notes:
        orig = converter.parse(str(original_path))
        synth = converter.parse(str(synthetic_path))
        
        
        o_notes = [(n.pitch.nameWithOctave, float(n.offset), float(n.quarterLength))
                   for n in orig.recurse().notes if n.isNote]
        s_notes = [(n.pitch.nameWithOctave, float(n.offset), float(n.quarterLength))
                   for n in synth.recurse().notes if n.isNote]
        results["notes_identical"] = {
            "passed": o_notes == s_notes,
            "detail": f"original={len(o_notes)} notes, synthetic={len(s_notes)} notes"}

    # Check 2: totals preserved
    oc, ow = scan_dynamics(original_path)
    sc, sw = scan_dynamics(synthetic_path)
    results["constant_total_preserved"] = {
        "passed": sum(oc.values()) == sum(sc.values()),
        "detail": f"original={sum(oc.values())}, synthetic={sum(sc.values())}"}
    results["wedge_total_preserved"] = {
        "passed": (ow["crescendo"] + ow["diminuendo"]
                   == sw["crescendo"] + sw["diminuendo"]
                   and ow["stop"] == sw["stop"]),
        "detail": (f"starts {ow['crescendo']+ow['diminuendo']}->"
                   f"{sw['crescendo']+sw['diminuendo']}, stops {ow['stop']}->{sw['stop']}")}

    # Check 3: synthetic markings == original markings transformed by changelog
    expected = dict(oc)
    exp_wedge = {"crescendo": ow["crescendo"], "diminuendo": ow["diminuendo"]}
    for ch in changelog:
        if ch["kind"] == "constant":
            expected[ch["from"]] -= 1
            expected[ch["to"]] = expected.get(ch["to"], 0) + 1
        else:
            exp_wedge[ch["from"]] -= 1
            exp_wedge[ch["to"]] += 1
    expected = {k: v for k, v in expected.items() if v > 0}
    got_wedge = {"crescendo": sw["crescendo"], "diminuendo": sw["diminuendo"]}
    results["changelog_applied_exactly"] = {
        "passed": expected == sc and exp_wedge == got_wedge,
        "detail": f"constants expected={expected}, got={sc}; wedges expected={exp_wedge}, got={got_wedge}"}

    # Check 4: at least one marking was actually corrupted
    results["at_least_one_corruption"] = {
        "passed": len(changelog) > 0,
        "detail": f"{len(changelog)} markings corrupted"}

    # Check 5: wedge structure not degraded (pre-existing quirks tolerated)
    ok_o, iss_o = check_wedge_balance(original_path)
    ok_s, iss_s = check_wedge_balance(synthetic_path)
    results["wedge_structure_not_degraded"] = {
        "passed": iss_s == iss_o,
        "detail": ("identical structure"
                   + ("" if ok_o else f" (pre-existing issues in original: {len(iss_o)})"))
                  if iss_s == iss_o else
                  f"NEW issues: {'; '.join(sorted(set(iss_s) - set(iss_o))[:3])}"}
    return results


# ----------------------------------------------------------------------
# Score-side timeline extraction (safe-offset version, verified earlier)
# ----------------------------------------------------------------------
def _safe_offset_in(el, score):
    try:
        return float(el.getOffsetInHierarchy(score))
    except Exception:
        m = el.getContextByClass("Measure")
        if m is not None:
            try:
                return float(m.getOffsetInHierarchy(score)) + float(el.offset)
            except Exception:
                pass
        return None


def extract_marking_timeline(score_path):
    score = converter.parse(str(score_path))
    flat = score.flatten()
    constants = sorted(
        [(float(d.offset), d.value)
         for d in flat.getElementsByClass(m21dynamics.Dynamic)
         if d.value in VELOCITY_MAP],
        key=lambda t: t[0])
    # Deduplicate coincident markings: piano scores can carry a different
    # marking per staff at the same offset (e.g. mp in the right hand, f in
    # the left). A single global velocity profile can only follow one, so we
    # keep the last at each offset - and, crucially, render and validation
    # then share the same canonical timeline.
    deduped = []
    for off, m in constants:
        if deduped and abs(off - deduped[-1][0]) < 1e-6:
            deduped[-1] = (off, m)
        else:
            deduped.append((off, m))
    constants = deduped
    wedges = []
    for sp in score.recurse().getElementsByClass(spanner.Spanner):
        if isinstance(sp, m21dynamics.Crescendo):
            kind = "crescendo"
        elif isinstance(sp, m21dynamics.Diminuendo):
            kind = "diminuendo"
        else:
            continue
        first, last = sp.getFirst(), sp.getLast()
        if first is None or last is None:
            continue
        s_off = _safe_offset_in(first, score)
        e_base = _safe_offset_in(last, score)
        if s_off is None or e_base is None:
            continue
        e = e_base + float(getattr(last, "quarterLength", 0.0))
        if e > s_off:
            wedges.append((s_off, e, kind))
    wedges.sort(key=lambda t: t[0])
    return constants, wedges


def remove_overlapping_wedges(wedges):
    clean, dropped = [], []
    current_end = -1.0
    for (start, end, kind) in wedges:
        if start < current_end:
            dropped.append({"start": start, "end": end, "kind": kind})
            continue
        clean.append((start, end, kind))
        current_end = end
    return clean, dropped


def build_velocity_profile(constants, wedges):
    """
    Faithful profile of the (already corrupted) score - no inversion here.
    MUSICAL SUSTAIN SEMANTICS: after a wedge ends, the reached level HOLDS
    until the next marking (a performer who crescendos stays loud until told
    otherwise) - it does not snap back to the pre-wedge level.
    Implemented as a single left-to-right pass over a merged event timeline.
    """
    def next_constant_after(offset):
        for off, m in constants:
            if off > offset:
                return VELOCITY_MAP[m]
        return None

    events = sorted(
        [(off, 0, m) for off, m in constants] +          # constants first at ties
        [(ws, 1, (ws, we, k)) for (ws, we, k) in wedges],
        key=lambda e: (e[0], e[1]))

    current = VELOCITY_MAP[DEFAULT_DYNAMIC]
    breakpoints = [(-1e12, current)]     # (position, level from here on)
    segments = []
    for e in events:
        if e[1] == 0:                                    # constant marking
            current = VELOCITY_MAP[e[2]]
            breakpoints.append((e[0], current))
        else:                                            # wedge
            ws, we, kind = e[2]
            v_start = current
            v_next = next_constant_after(ws)
            if v_next is None or (kind == "crescendo" and v_next <= v_start) \
                    or (kind == "diminuendo" and v_next >= v_start):
                delta = 20 if kind == "crescendo" else -20
                v_end = int(np.clip(v_start + delta, 1, 127))
            else:
                v_end = v_next
            segments.append((ws, we, kind, v_start, v_end))
            current = v_end                              # SUSTAIN the reached level
            # A constant marking that falls INSIDE or AT THE END of the wedge
            # span takes precedence after the wedge (the wedge resolves into
            # it); only sustain v_end if no constant lies in (ws, we].
            if not any(ws < off <= we for off, _ in constants):
                breakpoints.append((we, current))
            else:
                current = next(VELOCITY_MAP[m] for off, m in reversed(constants)
                               if ws < off <= we)
    breakpoints.sort(key=lambda b: b[0])

    def level_at(offset):
        lvl = breakpoints[0][1]
        for o, val in breakpoints:
            if o <= offset:
                lvl = val
            else:
                break
        return lvl

    def v(offset):
        for (ws, we, kind, v_start, v_end) in segments:
            if ws <= offset <= we:
                frac = (offset - ws) / (we - ws) if we > ws else 0.0
                return v_start + frac * (v_end - v_start)
        return level_at(offset)

    return v, segments


# ----------------------------------------------------------------------
# Render: synthetic XML -> MIDI (timing template = midi_score.mid)
# ----------------------------------------------------------------------
def render_xml_to_midi(synthetic_xml_path, midi_template_path, out_path,
                       jitter=4, seed=42):
    """
    Render the corrupted score's dynamics onto a copy of midi_score.mid.
    ONLY velocities are written; onsets/durations/pitches stay untouched.
    Per-note jitter (+/-jitter, default 4 per methodology) with the given
    seed; every synthetic example should be rendered with a DIFFERENT seed
    so repeated synthetics of one piece are not identical (#5).
    """
    rng = random.Random(seed)
    constants, wedges = extract_marking_timeline(synthetic_xml_path)
    wedges, dropped = remove_overlapping_wedges(wedges)
    v_of_offset, segments = build_velocity_profile(constants, wedges)

    pm = pretty_midi.PrettyMIDI(str(midi_template_path))

    def offset_to_time(off_ql):
        return pm.tick_to_time(int(round(off_ql * pm.resolution)))

    notes_changed = 0
    for inst in pm.instruments:
        for note in inst.notes:
            off_ql = pm.time_to_tick(note.start) / pm.resolution
            target = v_of_offset(off_ql)
            jit = rng.randint(-jitter, jitter) if jitter > 0 else 0
            new_v = int(np.clip(round(target + jit), 1, 127))
            if new_v != note.velocity:
                notes_changed += 1
            note.velocity = new_v

    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    pm.write(str(out_path))

    return {
        "segments": [{"start_ql": s, "end_ql": e, "kind": k,
                      "v_start": vs, "v_end": ve,
                      "start_s": offset_to_time(s), "end_s": offset_to_time(e)}
                     for (s, e, k, vs, ve) in segments],
        "constants": [{"offset_ql": o, "marking": m, "time_s": offset_to_time(o)}
                      for o, m in constants],
        "dropped_wedges": dropped,
        "notes_changed": notes_changed,
    }


# ----------------------------------------------------------------------
# MIDI validation: rendered file follows the CORRUPTED score's profile
# ----------------------------------------------------------------------
def validate_rendered_midi(template_path, rendered_path, meta,
                           tolerance=9, time_tol=1e-3, boundary_margin=0.1):
    o = pretty_midi.PrettyMIDI(str(template_path))
    s = pretty_midi.PrettyMIDI(str(rendered_path))
    on = sorted([n for i in o.instruments for n in i.notes],
                key=lambda n: (n.start, n.pitch))
    sn = sorted([n for i in s.instruments for n in i.notes],
                key=lambda n: (n.start, n.pitch))

    res = {}
    res["note_count"] = {"passed": len(on) == len(sn),
                         "detail": f"{len(on)} vs {len(sn)}"}
    if len(on) != len(sn):
        return res

    onset_diff = max((abs(a.start - b.start) for a, b in zip(on, sn)), default=0.0)
    dur_diff = max((abs((a.end - a.start) - (b.end - b.start))
                    for a, b in zip(on, sn)), default=0.0)
    res["onsets_unchanged"] = {"passed": onset_diff < time_tol,
                               "detail": f"max onset diff = {onset_diff:.2e}s"}
    res["durations_unchanged"] = {"passed": dur_diff < time_tol,
                                  "detail": f"max duration diff = {dur_diff:.2e}s"}
    res["pitches_unchanged"] = {
        "passed": all(a.pitch == b.pitch for a, b in zip(on, sn)),
        "detail": "all pitches identical"}

    wedge_spans = [(g["start_s"], g["end_s"], g["kind"], g["v_start"], g["v_end"])
                   for g in meta["segments"]]

    def in_wedge(t):
        return any(ws <= t <= we for ws, we, *_ in wedge_spans)

    # Constant regions hit the corrupted target
    bounds = [(c["time_s"], c["marking"]) for c in meta["constants"]]
    wedge_starts = sorted(ws for ws, we, *_ in wedge_spans)
    region_fail = []
    for i, (t0, marking) in enumerate(bounds):
        t1 = bounds[i + 1][0] if i + 1 < len(bounds) else max(
            (n.end for n in sn), default=t0)
        # The marking's level only governs until the first wedge AT or after it
        # (a wedge often begins exactly at a constant, e.g. "mf + crescendo");
        # after that wedge, the SUSTAINED level legitimately differs.
        next_ws = next((ws for ws in wedge_starts if ws >= t0 - 1e-9), None)
        if next_ws is not None:
            t1 = min(t1, next_ws)
        target_v = VELOCITY_MAP[marking]
        vs = [n.velocity for n in sn
              if t0 + boundary_margin <= n.start < t1 - boundary_margin
              and not in_wedge(n.start)]
        if len(vs) < 3:
            continue
        mean_v = float(np.mean(vs))
        if abs(mean_v - target_v) > tolerance:
            region_fail.append(f"{marking} @ {t0:.1f}s: mean {mean_v:.1f} vs {target_v}")
    res["constant_regions_on_target"] = {
        "passed": len(region_fail) == 0,
        "detail": "; ".join(region_fail[:3]) or
                  f"all constant regions within +/-{tolerance} of corrupted target"}

    # Wedge spans trend in the corrupted direction
    wedge_fail, skipped = [], 0
    for (ws, we, kind, v0, v1) in wedge_spans:
        pts = [(n.start, n.velocity) for n in sn if ws <= n.start <= we]
        # Skip spans where the slope direction is not statistically checkable:
        # very few notes, an intended ramp smaller than the jitter band
        # (|delta| < 2*jitter means noise can legitimately flip the fitted sign),
        # or a span too short in TIME for a slope fit to mean anything
        # (e.g. a 0.18s wedge over a single chord - no human crescendo fits there).
        if len(pts) < 5 or abs(v1 - v0) < 9 or (we - ws) < 0.5:
            skipped += 1
            continue
        t = np.array([p[0] for p in pts]); v = np.array([p[1] for p in pts])
        slope = np.polyfit(t, v, 1)[0]
        want_up = v1 > v0
        if (want_up and slope <= 0) or ((not want_up) and slope >= 0):
            wedge_fail.append(f"{kind} @ {ws:.1f}-{we:.1f}s slope={slope:.2f}")
    res["wedge_regions_follow_corruption"] = {
        "passed": len(wedge_fail) == 0,
        "detail": ("; ".join(wedge_fail[:3]) or "all checkable wedge spans trend as corrupted score says")
                  + (f" ({skipped} span(s) too short/flat to test)" if skipped else "")}
    return res


## 3. Load metadata and build the per-piece generation plan
`n_synth = max(1, round(min(real_count, CAP) × SYNTH_RATIO))` per piece (Problem 3);
full inversions only where a piece gets ≥2 synthetics (Problem 11); unique seed per example (Problem 5).

In [ ]:
# @title Generation plan
CAP        = 10      # matches the soft cap applied later at assembly
SYNTH_RATIO = 0.3    # target ~30% synthetic share per piece
FULL_INVERSION_SHARE = 0.25   # minority of extreme examples
WEDGE_FLIP_PROB = 0.5         # share of wedges flipped in shift-mode examples
# JITTER controls the human-wobble deviations (Problem 5). This is the ONE knob
# that produces the two synthetic SETS you compare downstream:
#   JITTER = 4  -> realistic wobble (the set you actually train on)
#   JITTER = 0  -> perfectly uniform velocities (deliberately-flawed ABLATION set,
#                  used to demonstrate that without jitter a model can cheat by
#                  detecting synthetic uniformity instead of learning adherence).
# Run this whole notebook once with each value; the output folder name changes
# automatically so the two sets never overwrite each other.
JITTER = 0
SEED = 42
SYNTH_VARIANT = 'nojitter' if JITTER > 0 else 'nojitter'
SYNTH_ROOT = BASE / f'synthetic_{SYNTH_VARIANT}'
print('Jitter setting:', JITTER, '-> writing to', SYNTH_ROOT)

clean = pd.read_csv(BASE / 'cleaned_metadata.csv')
real_counts = clean.groupby('folder').size()
pieces_df = clean.drop_duplicates('folder')[
    ['composer', 'folder', 'xml_score', 'midi_score']].reset_index(drop=True)
SYNTH_ROOT.mkdir(parents=True, exist_ok=True)

plan_rows = []
for idx, row in pieces_df.iterrows():
    real = int(real_counts[row.folder])
    capped_real = min(real, CAP)
    n_synth = max(1, round(capped_real * SYNTH_RATIO))
    if n_synth == 1:
        n_invert = 0                     # single-synthetic pieces: subtle only
    else:
        n_invert = min(n_synth - 1, max(1, round(n_synth * FULL_INVERSION_SHARE)))
    modes = ['shift'] * (n_synth - n_invert) + ['invert'] * n_invert
    for k, mode in enumerate(modes):
        plan_rows.append({'piece': row.folder, 'composer': row.composer,
                          'xml_score': row.xml_score, 'midi_score': row.midi_score,
                          'k': k, 'mode': mode,
                          'seed': SEED + idx * 100 + k,   # unique + reproducible
                          'real_count': real, 'capped_real': capped_real})
plan = pd.DataFrame(plan_rows)
print(f'{plan.piece.nunique()} pieces -> {len(plan)} synthetic examples planned')
print(plan['mode'].value_counts())
print('examples per piece:', plan.groupby('piece').size().value_counts().sort_index().to_dict())

Jitter setting: 0 -> writing to /content/drive/MyDrive/fyp_try2/synthetic_nojitter
187 pieces -> 271 synthetic examples planned
mode
shift     216
invert     55
Name: count, dtype: int64
examples per piece: {1: 132, 2: 26, 3: 29}


## 4. Coverage check
How many planned pieces actually contain dynamics — so a low output count is never a
mystery: it is the Bach urtext scores (Problem 10), not a bug.

In [ ]:
# @title Scan scores for dynamics (fast raw-XML scan)
N_PIECES = 20      # limit for a quick run; set None to process every piece

sel_pieces = pieces_df if N_PIECES is None else pieces_df.sample(N_PIECES, random_state=SEED)
has_dyn = {}
for row in sel_pieces.itertuples(index=False):
    try:
        oc, ow = scan_dynamics(ASAP_ROOT / row.xml_score)
        has_dyn[row.folder] = bool(oc) or bool(ow['crescendo'] or ow['diminuendo'])
    except Exception:
        has_dyn[row.folder] = False

n_with = sum(has_dyn.values())
print(f'{n_with} / {len(sel_pieces)} selected pieces have dynamics '
      f'({len(sel_pieces) - n_with} will be skipped - typically Bach urtext)')
plan_sel = plan[plan.piece.isin([p for p, ok in has_dyn.items() if ok])]
print(f'-> {len(plan_sel)} synthetic examples will be generated')

13 / 20 selected pieces have dynamics (7 will be skipped - typically Bach urtext)
-> 16 synthetic examples will be generated


## 5. Batch: corrupt XML → render MIDI → validate, per example

In [ ]:
# @title Run the full pipeline
xml_out  = SYNTH_ROOT / 'musicxml'
midi_out = SYNTH_ROOT / 'midi'

gen_log, val_rows = [], []
t0 = time.time()
for i, ex in enumerate(plan_sel.itertuples(index=False), 1):
    stem = f'synthetic_{ex.k:02d}_{ex.mode}'
    xml_dst  = xml_out / ex.piece / f'{stem}.musicxml'
    mani_dst = xml_out / ex.piece / f'{stem}_manifest.json'
    midi_dst = midi_out / ex.piece / f'{stem}.mid'
    rec = {'piece': ex.piece, 'k': ex.k, 'mode': ex.mode, 'seed': ex.seed}
    try:
        # 1. Corrupt the XML
        changelog = generate_synthetic_xml(ASAP_ROOT / ex.xml_score, xml_dst,
                                           seed=ex.seed, mode=ex.mode,
                                           wedge_flip_prob=WEDGE_FLIP_PROB)
        if not changelog:
            rec['status'] = 'skipped (nothing corruptible)'
            gen_log.append(rec); continue
        mani_dst.write_text(json.dumps(
            {'piece': ex.piece, 'mode': ex.mode, 'seed': ex.seed,
             'changes': changelog}, indent=1))

        # 2. Validate the XML (full note check only on the first example per piece)
        xres = validate_pair(ASAP_ROOT / ex.xml_score, xml_dst, changelog,
                             check_notes=(ex.k == 0))

        # 3. Render to MIDI (faithful render of the corrupted score, +/-4 jitter)
        meta = render_xml_to_midi(xml_dst, ASAP_ROOT / ex.midi_score, midi_dst,
                                  jitter=JITTER, seed=ex.seed)

        # 4. Validate the rendered MIDI
        mres = validate_rendered_midi(ASAP_ROOT / ex.midi_score, midi_dst, meta)

        allres = {**{f'xml_{k}': v for k, v in xres.items()},
                  **{f'midi_{k}': v for k, v in mres.items()}}
        vrow = {'piece': ex.piece, 'k': ex.k, 'mode': ex.mode,
                **{k: v['passed'] for k, v in allres.items()}}
        vrow['all_passed'] = all(v['passed'] for v in allres.values())
        val_rows.append(vrow)
        rec.update(status='generated', n_corruptions=len(changelog),
                   notes_rendered=meta['notes_changed'],
                   overlaps_removed=len(meta['dropped_wedges']))
    except Exception as e:
        rec['status'] = f'error: {type(e).__name__}: {e}'
    gen_log.append(rec)
    if i % 10 == 0:
        print(f'{i}/{len(plan_sel)} examples | {time.time()-t0:.0f}s elapsed')

gen_df, val_df = pd.DataFrame(gen_log), pd.DataFrame(val_rows)
print()
print(gen_df['status'].value_counts())
if len(val_df):
    print(f"{val_df['all_passed'].sum()} / {len(val_df)} examples passed ALL checks")
gen_df.to_csv(SYNTH_ROOT / 'generation_log.csv', index=False)
val_df.to_csv(SYNTH_ROOT / 'validation_log.csv', index=False)

/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/musicxml/xmlToM21.py:5491: MusicXMLWarning: Could not import pedal: Error in getting PedalMark
  warnings.warn(f'Could not import {tag}: {excep}', MusicXMLWarning)
/usr/local/lib/python3.12/dist-packages/music21/musicxml/xmlToM21.py:2240: MusicXMLWarning: Warning: measure 217 in part Pianois overfull: 241/120 > 2.0,assuming 2.0 is correct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/musicxml/xmlToM21.py:2240: MusicXMLWarning: Warning: measure 218 in part Pianois overfull: 329/160 > 2.0,assuming 2.0 is correct.
  warnings.warn(


10/16 examples | 97s elapsed


/usr/local/lib/python3.12/dist-packages/music21/musicxml/xmlToM21.py:2240: MusicXMLWarning: Warning: measure 31 in part Midi_1is overfull: 1153/480 > 2.0,assuming 2.0 is correct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/musicxml/xmlToM21.py:2240: MusicXMLWarning: Warning: measure 32 in part Midi_1is overfull: 1153/480 > 2.0,assuming 2.0 is correct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/musicxml/xmlToM21.py:2240: MusicXMLWarning: Warning: measure 33 in part Midi_1is overfull: 1153/480 > 2.0,assuming 2.0 is correct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/musicxml/xmlToM21.py:2240: MusicXMLWarning: Warning: measure 34 in part Midi_1is overfull: 1153/480 > 2.0,assuming 2.0 is correct.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/music21/musicxml/xmlToM21.py:2240: MusicXMLWarning: Warning: measure 35 in part Midi_1is overfull: 1153/480 > 2.0,assuming 2.0 is correct.
  warnings.warn(
/usr/local/lib/pytho


status
generated    16
Name: count, dtype: int64
16 / 16 examples passed ALL checks


## 6. Seed-independence spot check (Problem 5)
Two synthetics of the same piece must differ from each other.

In [ ]:
# @title Compare velocities between two synthetics of one piece
import pretty_midi
multi = val_df[val_df.k > 0]['piece'].unique() if len(val_df) else []
if len(multi):
    piece = multi[0]
    files = sorted((midi_out / piece).glob('synthetic_*.mid'))[:2]
    v = [[n.velocity for i in pretty_midi.PrettyMIDI(str(f)).instruments
          for n in i.notes] for f in files]
    diff = sum(1 for a, b in zip(*v) if a != b)
    print(f'{piece}: {diff}/{len(v[0])} note velocities differ between '
          f'{files[0].name} and {files[1].name}')
else:
    print('No piece received 2+ synthetics in this run - raise N_PIECES to check.')

Beethoven/Piano_Sonatas/24-1_no_2_repeat: 3030/3030 note velocities differ between synthetic_00_shift.mid and synthetic_01_invert.mid


## 7. Detailed HTML report with graphs

In [ ]:
# @title Helper: self-contained HTML report (figures embedded as base64)
import base64, io, datetime

def fig_to_b64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=110, bbox_inches='tight')
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()

def build_html_report(title, sections, out_path):
    parts = [f"<html><head><meta charset='utf-8'><title>{title}</title></head>"
             f"<body style='font-family:Arial;max-width:1000px;margin:auto'>"
             f"<h1>{title}</h1>"
             f"<p>Generated: {datetime.datetime.now().isoformat(timespec='seconds')}</p>"]
    for heading, content in sections:
        parts.append(f"<h2>{heading}</h2>")
        if isinstance(content, tuple) and content[0] == 'img':
            parts.append(f"<img src='data:image/png;base64,{content[1]}' style='max-width:100%'>")
        else:
            parts.append(content)
    parts.append('</body></html>')
    Path(out_path).write_text('\n'.join(parts), encoding='utf-8')
    print('Report written to:', out_path)


In [ ]:
# @title Build graphs + report
assert len(val_df), 'Nothing was generated - raise N_PIECES.'
check_cols = [c for c in val_df.columns if c not in ('piece', 'k', 'mode', 'all_passed', 'error')]

fig1, ax = plt.subplots(figsize=(9, 5))
rates = val_df[check_cols].mean().sort_values() * 100
rates.plot.barh(ax=ax, color=['#2a9d8f' if v == 100 else '#e76f51' for v in rates])
ax.set_xlabel('% of examples passing'); ax.set_xlim(0, 108)
ax.set_title('Validation pass rate per check (XML + rendered MIDI)')
for i, v in enumerate(rates): ax.text(v + 1, i, f'{v:.0f}%', va='center')
img1 = fig_to_b64(fig1)

gen_ok = gen_df[gen_df.status == 'generated']
fig2, axes = plt.subplots(1, 2, figsize=(12, 3.8))
gen_ok['mode'].value_counts().plot.bar(ax=axes[0], color=['#264653', '#e76f51'])
axes[0].set_title('Severity mix (Problem 11)'); axes[0].set_xlabel('')
gen_ok.groupby('piece').size().value_counts().sort_index().plot.bar(ax=axes[1], color='#e9c46a')
axes[1].set_title('Synthetics per piece (Problem 3)'); axes[1].set_xlabel('examples per piece')
plt.tight_layout()
img2 = fig_to_b64(fig2)

# Velocity comparison for one shift-mode and one invert-mode example
sample_imgs = []
for mode in ['shift', 'invert']:
    cand = gen_ok[gen_ok['mode'] == mode].head(1)
    if not len(cand):
        continue
    ex = cand.iloc[0]
    row = pieces_df[pieces_df.folder == ex.piece].iloc[0]
    stem = f"synthetic_{int(ex.k):02d}_{ex['mode']}"
    o = pretty_midi.PrettyMIDI(str(ASAP_ROOT / row.midi_score))
    s = pretty_midi.PrettyMIDI(str(midi_out / ex.piece / f'{stem}.mid'))
    on = sorted([n for i in o.instruments for n in i.notes], key=lambda n: n.start)
    sn = sorted([n for i in s.instruments for n in i.notes], key=lambda n: n.start)
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot([n.start for n in on], [n.velocity for n in on], '.', ms=3, alpha=.35, label='midi_score (template)')
    ax.plot([n.start for n in sn], [n.velocity for n in sn], '.', ms=3, alpha=.6, label=f'synthetic ({mode})')
    ax.set_xlabel('time (s)'); ax.set_ylabel('MIDI velocity'); ax.legend(loc='lower right')
    ax.set_title(f'{ex.piece} — {stem}')
    sample_imgs.append((f'Velocity: {mode} example — {ex.piece}', ('img', fig_to_b64(fig))))

summary = (f"<p><b>{len(val_df)}</b> examples across <b>{val_df.piece.nunique()}</b> pieces &middot; "
           f"<b>{val_df['all_passed'].sum()}</b> passed all checks "
           f"({100*val_df['all_passed'].mean():.1f}%)</p>"
           + val_df.head(50).to_html(index=False))
build_html_report('Synthetic Pipeline Report — Corrupted MusicXML → Rendered MIDI',
                  [('Summary', summary),
                   ('Pass rate per check', ('img', img1)),
                   ('Severity mix and per-piece counts', ('img', img2))] + sample_imgs,
                  SYNTH_ROOT / 'report_pipeline.html')

/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


Report written to: /content/drive/MyDrive/fyp_try2/synthetic_nojitter/report_pipeline.html


## 8. Experiments-appendix log entry

In [ ]:
# @title Log this run for the experiments appendix
exp_path = BASE / 'experiments_log.csv'
exp_df = pd.read_csv(exp_path) if exp_path.exists() else pd.DataFrame(
    columns=['experiment_no', 'date', 'approach', 'hypothesis', 'outcome', 'lessons'])

new_row = {
    'experiment_no': len(exp_df) + 1,
    'date': pd.Timestamp.now().strftime('%Y-%m-%d'),
    'approach': ('MusicXML->MIDI pipeline: severity mix '
                 f'(shift {1-FULL_INVERSION_SHARE:.0%} / invert {FULL_INVERSION_SHARE:.0%}), '
                 f'proportional counts (ratio {SYNTH_RATIO}), jitter +/-{JITTER} ({SYNTH_VARIANT} set), per-example seeds'),
    'hypothesis': ('One-level shifts near the decision boundary + minority inversions give '
                   'realistic negatives; proportional counts keep ~30% synthetic share per piece'),
    'outcome': f"{val_df['all_passed'].sum() if len(val_df) else 0}/{len(val_df)} examples passed all validation checks",
    'lessons': 'EDIT ME: add observations from this run',
}
exp_df = pd.concat([exp_df, pd.DataFrame([new_row])], ignore_index=True)
exp_df.to_csv(exp_path, index=False)
exp_df.tail()

,experiment_no,date,approach,hypothesis,outcome,lessons
4,5,2026-08-10,MusicXML->MIDI pipeline: severity mix (shift 7...,One-level shifts near the decision boundary + ...,16/16 examples passed all validation checks,EDIT ME: add observations from this run
5,6,2026-08-10,"Dataset assembly (jitter set, normalised): tim...",Realistic jitter forces the model to learn adh...,"constants: 15805 rows, wedges: 11392 rows, Gro...",EDIT ME
6,7,2026-08-10,MusicXML->MIDI pipeline: severity mix (shift 7...,One-level shifts near the decision boundary + ...,16/16 examples passed all validation checks,EDIT ME: add observations from this run
7,8,2026-08-10,"Dataset assembly (nojitter set, normalised): t...",No-jitter ablation: expected to score deceptiv...,"constants: 15805 rows, wedges: 11392 rows, Gro...",EDIT ME
8,9,2026-08-10,MusicXML->MIDI pipeline: severity mix (shift 7...,One-level shifts near the decision boundary + ...,16/16 examples passed all validation checks,EDIT ME: add observations from this run
